In [ ]:
import os
import sys
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

repo_url = "https://github.com/hoangquan1503/Topic_classification.git" 
repo_name = "Topic_classification"

# if not exist
if not os.path.exists(repo_name):
    print("Cloning repository...")
    !git clone {repo_url}


%cd {repo_name}
print(f"Current: {os.getcwd()}")

!git pull

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

if 'src' in sys.modules:
    del sys.modules['src']
    print("remove duplicate src")

Mounted at /content/drive
Cloning repository...
Cloning into 'Topic_classification'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 12 (delta 1), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 269.90 KiB | 5.74 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Topic_classification
Current: /content/Topic_classification
Already up to date.


In [ ]:
from huggingface_hub import login
login(token="hf_") 

In [ ]:
from datasets import load_dataset
ds = load_dataset("UniverseTBD/arxiv-abstracts-large")

ds

README.md:   0%|          | 0.00/810 [00:00<?, ?B/s]

arxiv-metadata-oai-snapshot.json: reconstructing file:   0%|          |  0.00B / 3.82GB            

arxiv-metadata-oai-snapshot.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2292057 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed'],
        num_rows: 2292057
    })
})

In [6]:
print(ds['train'][0]['abstract'])
print(ds['train'][0]['categories'])

  A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from quark-antiquark,
gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as
all-orders resummation of initial-state gluon radiation valid at
next-to-next-to-leading logarithmic accuracy. The region of phase space is
specified in which the calculation is most reliable. Good agreement is
demonstrated with data from the Fermilab Tevatron, and predictions are made for
more detailed tests with CDF and DO data. Predictions are shown for
distributions of diphoton pairs produced at the energy of the Large Hadron
Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs
boson are contrasted with those produced from QCD processes at the LHC, showing
that enhanced sensitivity to the signal can be obtained with judicious
selection of events.

hep-ph


In [ ]:
all_categories = ds['train']['categories']
unique_cate = set()

# Split name of categories 
for category in all_categories:
    topic = category.split(' ')[0]
    topic = topic.split('.')[0] #only take first general topic
    unique_cate.add(topic)
    
print(unique_cate)
print(len(unique_cate))
        

{'cs', 'alg-geom', 'atom-ph', 'funct-an', 'chem-ph', 'math-ph', 'astro-ph', 'nlin', 'chao-dyn', 'solv-int', 'stat', 'physics', 'ao-sci', 'cond-mat', 'q-fin', 'nucl-th', 'math', 'patt-sol', 'hep-ex', 'econ', 'acc-phys', 'dg-ga', 'gr-qc', 'q-bio', 'q-alg', 'comp-gas', 'hep-lat', 'hep-ph', 'eess', 'adap-org', 'mtrl-th', 'supr-con', 'hep-th', 'cmp-lg', 'plasm-ph', 'bayes-an', 'nucl-ex', 'quant-ph'}
38


In [ ]:
from src.data_preprocessing import processed_ds, preprocess
from datasets import load_from_disk

# Save to Drive
save_path = '/content/drive/MyDrive/Topic_classification/processed_train_dataset' 
if os.path.exists(save_path):
    processed_sample = load_from_disk(save_path)
else:
    processed_sample = processed_ds(ds['train'], preprocess, num_proc=4)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    processed_sample.save_to_disk(save_path)
    print('save done')
    


    

In [9]:
print(processed_sample[:2])

{'id': ['0704.0001', '0704.0002'], 'submitter': ['Pavel Nadolsky', 'Louis Theran'], 'authors': ["C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan", 'Ileana Streinu and Louis Theran'], 'title': ['Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies', 'Sparsity-certifying Graph Decompositions'], 'comments': ['37 pages, 15 figures; published version', 'To appear in Graphs and Combinatorics'], 'journal-ref': ['Phys.Rev.D76:013009,2007', None], 'doi': ['10.1103/PhysRevD.76.013009', None], 'report-no': ['ANL-HEP-PR-07-12', None], 'categories': ['hep-ph', 'math'], 'license': [None, 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/'], 'abstract': ['a fully differential calculation in perturbative quantum chromodynamics ispresented for the production of massive photon pairs at hadron colliders allnexttoleading order perturbative contributions from quarkantiquarkgluonantiquark and gluongluon subprocesses are included as well asallorders resummation of

In [ ]:
# Build dict to convert label to num
label_to_id = {label : i for i, label in enumerate(sorted(list(unique_cate)))}
id_to_label = {i : label  for i, label in enumerate(sorted(list(unique_cate)))}

print(label_to_id)



{'cs': 0, 'alg-geom': 1, 'atom-ph': 2, 'funct-an': 3, 'chem-ph': 4, 'math-ph': 5, 'astro-ph': 6, 'nlin': 7, 'chao-dyn': 8, 'solv-int': 9, 'stat': 10, 'physics': 11, 'ao-sci': 12, 'cond-mat': 13, 'q-fin': 14, 'nucl-th': 15, 'math': 16, 'patt-sol': 17, 'hep-ex': 18, 'econ': 19, 'acc-phys': 20, 'dg-ga': 21, 'gr-qc': 22, 'q-bio': 23, 'q-alg': 24, 'comp-gas': 25, 'hep-lat': 26, 'hep-ph': 27, 'eess': 28, 'adap-org': 29, 'mtrl-th': 30, 'supr-con': 31, 'hep-th': 32, 'cmp-lg': 33, 'plasm-ph': 34, 'bayes-an': 35, 'nucl-ex': 36, 'quant-ph': 37}


In [ ]:
def convert_label(ds):
    ds['label_id'] = label_to_id[ds['label']]
    return ds

processed_sample = processed_sample.map(convert_label)

Map:   0%|          | 0/2292057 [00:00<?, ? examples/s]

In [14]:
dataset_split = processed_sample.train_test_split(test_size=0.2, seed=42, stratify_by_column='label_id')
X_train = dataset_split['train']
X_test = dataset_split['test']
print(len(X_train))

ValueError: Stratifying by column is only supported for ClassLabel column, and column label_id is Value.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from src.embedding_vector import EmbeddingVectorizer

bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf =  TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

embeddings = EmbeddingVectorizer()
X_train_embeddings = embeddings.transform(X_train)
X_test_embeddings = embeddings.transform(X_test)

# convert all to numpy for consistency
X_train_bow, X_train_bow = np.array(X_train_bow), np.array(X_test_bow)
X_train_tfidf, X_test_tfidf = np.array(X_train_tfidf), np.array(X_test_tfidf)

print(f'shape of BoW: {X_train_bow.shape}')
print(f'shape of tfidf: {X_train_tfidf}')
print(f'shape of embedding: {X_train_embeddings}')
